# 🧠 Notebook 00: Numeric Foundations

## 1. Purpose + Scope

This notebook establishes the foundational principles of the T81 numeric substrate. It explores:

*   **Balanced Ternary Principles**: Understanding the trinary digit (-1, 0, +1) system.
*   **T81Int Bit/Trit Layout**: How ternary values are encoded efficiently.
*   **Arithmetic Invariants**: Properties preserved across operations.
*   **Deterministic Overflow Behavior**: How T81 handles values exceeding capacity.
*   **Canonical Serialization**: Ensuring bit-for-bit identical storage.

## 2. Spec References

*   `spec/t81-data-types.md`
*   `spec/determinism-profile.md`
*   `include/t81/core/T81Int.hpp`

## 3. Determinism Tier

**Tier A (Strict Determinism)**: All operations shown here are strictly deterministic and platform-independent.

## 4. Reproducibility Setup

Ensure `t81_python` is built and available in `PYTHONPATH`.

In [1]:
import sys
import os

# Ensure the build directory is in the path
build_dir = os.path.abspath(os.path.join(os.getcwd(), "../build"))
if build_dir not in sys.path:
    sys.path.append(build_dir)

try:
    import t81_python
    print("✅ t81_python module loaded successfully.")
except ImportError:
    print("❌ Failed to load t81_python. Please build the project first.")
    sys.exit(1)

✅ t81_python module loaded successfully.


## 5. Exploratory Code: T81Int Basics

`T81Int<81>` is the fundamental integer type, representing a signed integer within a fixed trinary capacity. It is designed to model hardware-native balanced ternary integers.

In [2]:
from t81_python import T81Int

# Constructing T81Ints
a = T81Int(42)
b = T81Int(-5)
c = T81Int(0)

print(f"a = {a}")
print(f"b = {b}")
print(f"c = {c}")

# Basic Arithmetic
sum_val = a + b
diff_val = a - b
prod_val = a * b

print(f"{a} + {b} = {sum_val}")
print(f"{a} - {b} = {diff_val}")
print(f"{a} * {b} = {prod_val}")

a = <t81.T81Int value=42>
b = <t81.T81Int value=-5>
c = <t81.T81Int value=0>
<t81.T81Int value=42> + <t81.T81Int value=-5> = <t81.T81Int value=37>
<t81.T81Int value=42> - <t81.T81Int value=-5> = <t81.T81Int value=47>
<t81.T81Int value=42> * <t81.T81Int value=-5> = <t81.T81Int value=-210>


### 5.1 Limits and Overflow

T81Int has fixed limits. Let's inspect them.

In [3]:
max_val = T81Int.max_value()
min_val = T81Int.min_value()

print(f"Max Value: {max_val}")
print(f"Min Value: {min_val}")

# Note: Direct overflow testing via Python bindings might be tricky if the wrapper 
# enforces int64_t limits on input, but let's see the defined boundaries.
# In strict mode, overflow should trap or wrap deterministically depending on configuration.

Max Value: <t81.T81Int value=(large)>
Min Value: <t81.T81Int value=(large)>


## 6. Serialization Inspection

T81 ensures canonical serialization. While the Python repr gives a debug view, the internal bit/trit pattern is fixed.

In [4]:
# In a full implementation, we would inspect the raw bytes or trit-stream here.
# For now, we rely on the string representation as a proxy for value identity.
print(f"Canonical Repr of 42: {repr(T81Int(42))}")

Canonical Repr of 42: <t81.T81Int value=42>


## 7. Trace or Hash Validation

We verify that a sequence of operations produces the expected result hash.

In [5]:
import hashlib

def calculate_sequence_hash(n):
    val = T81Int(0)
    hasher = hashlib.sha256()
    for i in range(n):
        val = val + T81Int(i)
        hasher.update(str(val).encode('utf-8'))
    return hasher.hexdigest(), val

seq_hash, final_val = calculate_sequence_hash(100)
print(f"Sequence Hash (n=100): {seq_hash}")
print(f"Final Value: {final_val}")

# Expected deterministic hash for this sequence
# This value serves as a regression test anchor.
assert "value=4950" in str(final_val)

Sequence Hash (n=100): afccc3b18a9d402f339b564a60e2c8a265957247a9ceb4e28f565e18ae9fc755
Final Value: <t81.T81Int value=4950>


## 8. Failure Mode Demonstration

Demonstrating controlled failure or boundary conditions.

In [6]:
try:
    # Attempt to create a T81Int from a value outside representable range (conceptually)
    # Since we are bound to int64, we can try the int64 limits if T81Int<81> is smaller,
    # but T81Int<81> is actually quite large. 
    # Instead, let's just show standard error handling for type mismatches.
    invalid = T81Int("not a number")
except TypeError as e:
    print(f"Caught expected error: {e}")

Caught expected error: __init__(): incompatible constructor arguments. The following argument types are supported:
    1. t81_python.T81Int(arg0: typing.SupportsInt)

Invoked with: 'not a number'


## 9. Architectural Commentary

The `T81Int` type avoids the pitfalls of two's complement asymmetry. The balanced ternary representation ensures that negation is a simple trit-inversion, and there is no "negative zero". This symmetry simplifies algebraic properties and reduces edge cases in numerical algorithms.

## 10. Open Questions / Research Surface

*   **Hardware Acceleration**: How effectively can `T81Int` operations be mapped to FPGA logic cells optimized for ternary logic?
*   **Mixed-Precision Arithmetic**: Optimizing interactions between `T81Int<81>` and larger types.